In [64]:
import pandas as pd

In [65]:
price_dataset = pd.read_csv("../data/raw/NL/Netherlands.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [66]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Netherlands,NLD,2015-01-01 00:00:00,2015-01-01 01:00:00,36.56,2015-01-01 00:00:00
1,Netherlands,NLD,2015-01-01 01:00:00,2015-01-01 02:00:00,36.56,2015-01-01 01:00:00
2,Netherlands,NLD,2015-01-01 02:00:00,2015-01-01 03:00:00,36.56,2015-01-01 02:00:00
3,Netherlands,NLD,2015-01-01 03:00:00,2015-01-01 04:00:00,36.56,2015-01-01 03:00:00
4,Netherlands,NLD,2015-01-01 04:00:00,2015-01-01 05:00:00,36.56,2015-01-01 04:00:00


In [67]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [68]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,36.56,2015-01-01 00:00:00
1,36.56,2015-01-01 01:00:00
2,36.56,2015-01-01 02:00:00
3,36.56,2015-01-01 03:00:00
4,36.56,2015-01-01 04:00:00


In [69]:
solar_dataset = pd.read_csv("../data/raw/NL/solar-raw.csv")

In [70]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [71]:
solar_dataset = pd.read_csv("../data/raw/NL/solar-raw.csv")

In [72]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [73]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [74]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,0.0
1,2022-01-01 01:00:00+00:00,0.0
2,2022-01-01 02:00:00+00:00,0.0
3,2022-01-01 03:00:00+00:00,0.0
4,2022-01-01 04:00:00+00:00,0.0


In [75]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [76]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [77]:
meteo_dataset = pd.read_csv("../data/raw/NL/meteo-raw-netherlands.csv")

In [78]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,10.7,17.3,234,100,0.0,0.0
1,2022-01-01T01:00,10.9,16.8,235,100,0.0,0.0
2,2022-01-01T02:00,10.7,18.0,233,86,0.0,0.0
3,2022-01-01T03:00,10.3,16.1,230,21,0.0,0.0
4,2022-01-01T04:00,10.2,16.5,224,100,0.0,0.0


In [79]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [80]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [81]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [82]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,10.7,17.3,234,100,0.0,0.0,2022-01-01 00:00:00
1,10.9,16.8,235,100,0.0,0.0,2022-01-01 01:00:00
2,10.7,18.0,233,86,0.0,0.0,2022-01-01 02:00:00
3,10.3,16.1,230,21,0.0,0.0,2022-01-01 03:00:00
4,10.2,16.5,224,100,0.0,0.0,2022-01-01 04:00:00


In [83]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,124.70,2022-01-01 00:00:00,0.0,10.7,17.3,234,100,0.0,0.0
1,134.00,2022-01-01 01:00:00,0.0,10.9,16.8,235,100,0.0,0.0
2,58.80,2022-01-01 02:00:00,0.0,10.7,18.0,233,86,0.0,0.0
3,37.67,2022-01-01 03:00:00,0.0,10.3,16.1,230,21,0.0,0.0
4,39.70,2022-01-01 04:00:00,0.0,10.2,16.5,224,100,0.0,0.0


In [84]:
merged.to_csv("../data/processed/netherlands_merged.csv", index=False)